# 📖 Notebook 2 — Synchronous vs Asynchronous Replication

In Notebook 1 we saw a primary copy writes to a replica. But *when* does the primary
tell the client "done!"? That single decision is one of the most important tradeoffs in
distributed systems.

This notebook is a **pure-Python simulation**. We don't need Docker here — we'll build
a tiny toy database so you can see exactly what happens when the replica is a little
behind.

## Learning objectives

- Explain the three strategies: **synchronous**, **asynchronous**, and **semi-synchronous**.
- Feel the "stale read" problem: reading from a replica that hasn't caught up yet.
- Reason about the tradeoff between **durability**, **latency**, and **availability**.


## 🛠️ Setup

No Docker required for this notebook — it runs entirely in Python.

Make sure you've run `uv sync` from this lab's folder and selected the `.venv` kernel
in VS Code's kernel picker (top-right). If the kernel doesn't appear, reload the
window: `Cmd+Shift+P` → "Reload Window".


## 1. The three replication strategies in plain words

Imagine you wrote "update user 42's email" on the primary. The primary has to decide:

> *"When do I tell the client the write succeeded?"*

| Strategy | When does the primary answer the client? | What if the primary dies right after? |
|---|---|---|
| **Async** | Immediately after writing its own local disk. Replicas catch up later. | You might lose the write — the replica may not have received it yet. |
| **Sync** | Only after **every** replica confirms it wrote the change too. | Safe — at least one replica has it. But slow, and if any replica is down, writes stop. |
| **Semi-sync** | After **at least one** replica confirms. | Good middle ground — safe against single-node loss without waiting for everyone. |

A useful way to think about it: **async trades safety for speed; sync trades speed for
safety; semi-sync tries to have most of both.**


## 2. A toy replicated database

We'll model a primary and a replica as two tiny Python objects. Writes go through a
`Replicator` that, depending on its mode, either:

- pushes the write to the replica *before* returning (sync), or
- schedules the push on a background thread and returns immediately (async).

We'll then measure how long writes take and whether reads on the replica can return
stale data.


In [1]:
import threading
import time
from dataclasses import dataclass, field
from typing import Literal

from pydantic import BaseModel


class Node:
    """A toy in-memory key/value store that pretends to be a database node."""
    def __init__(self, name: str):
        self.name = name
        self.store: dict[str, str] = {}
        self._lock = threading.Lock()

    def write(self, key: str, value: str) -> None:
        # Simulate disk fsync: a small but nonzero cost.
        time.sleep(0.005)
        with self._lock:
            self.store[key] = value

    def read(self, key: str) -> str | None:
        with self._lock:
            return self.store.get(key)


class WriteResult(BaseModel):
    mode: Literal["sync", "async", "semi-sync"]
    latency_ms: float
    durably_on_replicas: int


class Replicator:
    def __init__(self, primary: Node, replicas: list[Node], mode: str, replica_delay_ms: int = 40):
        assert mode in {"sync", "async", "semi-sync"}
        self.primary = primary
        self.replicas = replicas
        self.mode = mode
        self.replica_delay_ms = replica_delay_ms

    def _ship_to(self, replica: Node, key: str, value: str) -> None:
        # Simulate network latency + replay delay between primary and replica.
        time.sleep(self.replica_delay_ms / 1000)
        replica.write(key, value)

    def write(self, key: str, value: str) -> WriteResult:
        start = time.perf_counter()
        self.primary.write(key, value)
        durable = 0

        if self.mode == "sync":
            # Wait for ALL replicas to apply the write.
            for r in self.replicas:
                self._ship_to(r, key, value)
                durable += 1
        elif self.mode == "semi-sync":
            # Wait for AT LEAST ONE replica, then background-ship to the rest.
            if self.replicas:
                self._ship_to(self.replicas[0], key, value)
                durable = 1
            for r in self.replicas[1:]:
                threading.Thread(target=self._ship_to, args=(r, key, value), daemon=True).start()
        else:  # async
            # Fire-and-forget to every replica.
            for r in self.replicas:
                threading.Thread(target=self._ship_to, args=(r, key, value), daemon=True).start()

        elapsed_ms = (time.perf_counter() - start) * 1000
        return WriteResult(mode=self.mode, latency_ms=round(elapsed_ms, 2), durably_on_replicas=durable)


## 3. The stale-read problem (async)

Here's the scenario every replicated system has to deal with:

1. Client A writes `email = new@...` on the primary.
2. The primary answers *"done"* immediately (async).
3. Client A's next request happens to be a read — and a load balancer routes it to a
   replica that has not yet received the change.
4. Client A sees their **old** email. 🤨

Let's make it happen on purpose:


In [2]:
primary = Node("primary")
replica = Node("replica-1")
repl = Replicator(primary=primary, replicas=[replica], mode="async", replica_delay_ms=50)

# Seed initial state on both nodes.
primary.write("user:42:email", "old@example.com")
replica.write("user:42:email", "old@example.com")

result = repl.write("user:42:email", "new@example.com")
print("write result:", result)

# Read immediately — before the replica has had time to catch up.
print("read from primary:", primary.read("user:42:email"))
print("read from replica:", replica.read("user:42:email"))  # probably STALE!

time.sleep(0.1)  # give the background shipper time to deliver
print("read from replica (after wait):", replica.read("user:42:email"))


write result: mode='async' latency_ms=45.09 durably_on_replicas=0
read from primary: new@example.com
read from replica: old@example.com


read from replica (after wait): new@example.com


You should see the primary return `new@example.com` right away, while the replica is
still reporting `old@example.com` for a brief moment. That is **replication lag**, and
it is the price you pay for async's speed.

### Why teams still choose async

- The write latency is **whatever the primary's local disk takes** — no network round
  trip to the replica. That matters a lot when users are waiting for an API response.
- If a replica is slow or temporarily offline, writes keep working.

The job of the application is to know when stale reads are OK (a timeline, a product
listing) and when they are not (*"show me my own profile immediately after I edited
it"* — typically routed back to the primary, or served from a cache the client
controls).


## 4. Measure the latency cost

Let's run the same 50 writes three times, once per mode, and see how long each mode
takes.


In [3]:
def benchmark(mode: str, n_writes: int = 50) -> dict:
    p = Node("primary")
    rs = [Node(f"replica-{i}") for i in range(3)]
    r = Replicator(primary=p, replicas=rs, mode=mode, replica_delay_ms=20)

    start = time.perf_counter()
    for i in range(n_writes):
        r.write(f"k{i}", f"v{i}")
    total_ms = (time.perf_counter() - start) * 1000
    return {"mode": mode, "writes": n_writes, "total_ms": round(total_ms, 1),
            "avg_per_write_ms": round(total_ms / n_writes, 2)}


for mode in ["async", "semi-sync", "sync"]:
    print(benchmark(mode))


{'mode': 'async', 'writes': 50, 'total_ms': 1813.2, 'avg_per_write_ms': 36.26}


{'mode': 'semi-sync', 'writes': 50, 'total_ms': 6837.8, 'avg_per_write_ms': 136.76}


{'mode': 'sync', 'writes': 50, 'total_ms': 19491.6, 'avg_per_write_ms': 389.83}


On a quiet machine you should see something like:

- **async** ≈ fastest per write (only local disk)
- **semi-sync** ≈ roughly one network hop slower (waits for *one* replica)
- **sync** ≈ slowest (waits for *all* replicas)

The exact numbers don't matter — what matters is the shape of the tradeoff.

## 5. Durability: what if the primary dies right after the write?

Latency isn't the only thing mode changes — it also changes **how much data you lose**
when the primary dies right after acknowledging a write.


In [4]:
def simulate_crash(mode: str) -> dict:
    p = Node("primary")
    rs = [Node(f"replica-{i}") for i in range(3)]
    r = Replicator(primary=p, replicas=rs, mode=mode, replica_delay_ms=30)

    r.write("order:1", "pending")

    # Simulate "primary catches fire" the instant it acknowledges the write.
    # We do NOT sleep — async replication may not have delivered anything yet.
    surviving_copies = sum(1 for rep in rs if rep.read("order:1") == "pending")
    return {"mode": mode, "replicas_with_the_write": surviving_copies}


for mode in ["async", "semi-sync", "sync"]:
    print(simulate_crash(mode))


{'mode': 'async', 'replicas_with_the_write': 0}


{'mode': 'semi-sync', 'replicas_with_the_write': 1}


{'mode': 'sync', 'replicas_with_the_write': 3}


## 6. Bad practice → best practice: *read your own writes*

The most common bug caused by async replication is also the most embarrassing:

> *A user updates their profile picture. The next page load shows the **old** picture
> because the read was served by a replica that hadn't caught up yet.*

Let's recreate the bug, then fix it. This is the heart of the **read-your-own-writes
(RYW) consistency** pattern.

### ❌ Bad practice: round-robin reads to any replica


In [5]:
# Build a tiny app: 1 primary + 2 replicas, async replication, 60 ms lag.
import itertools

primary = Node("primary")
replicas = [Node("replica-a"), Node("replica-b")]
repl = Replicator(primary=primary, replicas=replicas, mode="async", replica_delay_ms=60)

primary.write("user:7:avatar", "old.png")
for r in replicas:
    r.write("user:7:avatar", "old.png")

# A naive load balancer: round-robin across all 3 nodes.
all_nodes = [primary, *replicas]
rr = itertools.cycle(all_nodes)


def naive_read(key: str) -> str | None:
    node = next(rr)
    return node.read(key)


# User uploads new avatar (write is async; primary acks immediately).
repl.write("user:7:avatar", "new.png")

# User's browser immediately re-fetches their profile a few times.
print("naive reads right after the write:")
for _ in range(4):
    print(" ", naive_read("user:7:avatar"))


naive reads right after the write:
  new.png
  old.png
  old.png
  new.png


You'll usually see a mix of `new.png` and `old.png` — the user sees their *own*
update flicker between fresh and stale. Awful UX.

### ✅ Best practice: route a user's reads to the primary for a short window after their write

The fix is dead simple: when a user writes, remember *when* they wrote, and for the
next N seconds route **their** reads back to the primary (or to a node you know has
caught up). Other users keep enjoying the cheap replica reads.


In [6]:
class RYWRouter:
    """Route reads to the primary for `sticky_ms` after a user's last write."""
    def __init__(self, primary: Node, replicas: list[Node], sticky_ms: int = 200):
        self.primary = primary
        self.replicas = itertools.cycle(replicas)
        self.sticky_ms = sticky_ms
        self._last_write_at: dict[str, float] = {}

    def write(self, user_id: str, key: str, value: str) -> None:
        primary.write(key, value)  # in real life: go through the Replicator
        self._last_write_at[user_id] = time.perf_counter()

    def read(self, user_id: str, key: str) -> str | None:
        last = self._last_write_at.get(user_id, 0)
        recently_wrote = (time.perf_counter() - last) * 1000 < self.sticky_ms
        node = self.primary if recently_wrote else next(self.replicas)
        return node.read(key)


# Same setup, but route reads through RYWRouter.
primary = Node("primary")
replicas = [Node("replica-a"), Node("replica-b")]
for n in (primary, *replicas):
    n.write("user:7:avatar", "old.png")

router = RYWRouter(primary=primary, replicas=replicas, sticky_ms=200)
router.write(user_id="7", key="user:7:avatar", value="new.png")

print("RYW reads right after the write:")
for _ in range(4):
    print(" ", router.read(user_id="7", key="user:7:avatar"))


RYW reads right after the write:
  new.png
  new.png
  new.png
  new.png


All four reads now return `new.png`. The user always sees their own writes — and once
the sticky window expires, their reads go back to the cheap replicas.

Variations you'll see in real systems:

- **Session sticky bit / cookie** — the load balancer pins the user to the primary
  for a few seconds after a `POST`/`PUT`.
- **Wait-for-LSN** — the app remembers the WAL position of the write and asks a
  replica *"are you caught up to LSN X?"* before reading from it (Postgres
  `pg_last_wal_replay_lsn()`).
- **Causal tokens** — MongoDB and FoundationDB hand out a token after each write; the
  client sends it on the next read so the server can wait for that version.

## 7. Real-world examples

| System | Replication model | Why they chose it |
|---|---|---|
| **Twitter (X) timeline** | Async — writes fan out to many caches and replicas. | A 1–2 second delay on someone else's tweet is fine. |
| **Stripe payments** | Sync to multiple zones before the API returns. | Losing an acked payment is unacceptable. |
| **Postgres on RDS Multi-AZ** | Synchronous to a hot standby in another AZ. | Required for the 99.95% SLA they advertise. |
| **MySQL semi-sync** at GitHub/Booking | Wait for **one** replica ack. | Survives a single-machine fire without paying full sync cost. |
| **Kafka with `acks=all`** | Sync to all in-sync replicas. | Same idea, applied to a log instead of a database. |

> Rule of thumb: **start with semi-sync**. Switch to full sync only when losing the
> last write would be a business catastrophe; switch to async only when the workload
> is genuinely loss-tolerant (analytics, telemetry).


Typical result:

- **async** → `0` replicas have the write. If the primary is gone, the write is gone too.
- **semi-sync** → `1` replica has it, enough to recover from.
- **sync** → all replicas have it.

### How should you choose?

- **Picking a username? Analytics event? Tweet?** Async is usually fine — speed wins,
  and a tiny chance of losing the most recent write is acceptable.
- **Payment confirmation? Inventory decrement?** Semi-sync is the pragmatic default:
  you get durability on at least one other machine *and* acceptable latency.
- **Sync** is rare in real systems because one slow replica stalls every write — great
  for correctness demos, painful in production.

## 8. Recap

- **Async**: fast, but a brief window where replicas are behind. You can read stale data.
- **Sync**: safe, but slow and fragile — one slow replica slows everyone.
- **Semi-sync**: the usual production choice. Fast, and the data survives single-node loss.
- Replication lag is a real thing you need to design around — not a bug to remove.

### What's next

Notebook 3 zooms out one more level. What if there's no primary at all, and every read
and write has to ask a **quorum** of nodes?
